# Search for spliceosome complex component homologs in ciliates

This protocol is informed by https://onlinelibrary.wiley.com/doi/10.1111/jeu.12844 and https://www.ncbi.nlm.nih.gov/pmc/articles/PMC10158995/. However instead of a manually curated list of spliceosomal components, we pull reviewed annotations directly from Uniprot and OrthoDB through their SPARQL endpoint and/or API. This has the advantage of being faster and less error prone, and more reproducible. Disadvantage is that we cannot include unpublished data.

 * From Uniprot, get the list of proteins annotated with the GO term GO:0005681 (spliceosomal complex) in yeast, the model organism for most studies on the spliceosome, and where the annotation has been reviewed (Swissprot). Because annotations are inconsistent and patchy in most organisms, we choose to build a shortlist based on yeast proteins and then search for orthologs in our species of interest, rather than relying on the annotations in our species of interest.
 * Get the associated OrthoDB orthogroup for each protein.
 * Find the orthologs annotated by OrthoDB from ciliate genomes. NB: only seven ciliate genomes are included in OrthoDB:
     * Oxytricha trifallax
     * Pseudocohnilembus persalinus
     * Paramecium tetraurelia strain d4-2
     * Ichthyophthirius multifiliis
     * Stylonychia lemnae
     * Stentor coeruleus
     * Tetrahymena thermophila SB210
 * Search for homologs in Mesodinium rubrum (TODO)
     * Map predicted proteins to OrthoDB with orthomapper, and see if there are any hits to the orthogroups of interest
     * De novo ortholog clustering with OrthoFinder, and see whether the proteins of interest are clustered with those identified in other ciliate species
     * Blastx searches of the genome nucleotide sequence, in case the proteins were missed by gene prediction errors. How to distinguish against false positives? We could compare the E-values of hits against other proteins in the query genome which are positively grouped with a Mesodinium ortholog by OrthoFinder or OrthoMapper.
     * Search Mesodinium predicted proteins for any Pfam domains that are also found in spliceosomal components, based on the Pfam domains in ciliate spliceosomal components.

How did Wong et al. search for homologs to spliceosomal proteins in Porphyridium purpureum? (see their Supplementary Methods)

 * Reciprocal-best-hit blastp with spliceosomal proteins from human, yeast, Cyanidioschyzon merolae, and Arabidopsis
 * Filtered reciprocal-best-hit blastp results: >25% identity, >50% length coverage, E-value <1e-6
 * To identify more divergent homologs, related cutoffs: >20% identity, >??% length coverage
 * Repeated reciprocal-best-hit search with blastx and tblastn, in case some genes were not correctly predicted in the target genome

Comments to the above:

 * The set of query sequences used for reciprocal-best-hit should be the whole predicted proteome, rather than a shortlist of proteins, to have meaningful E-values, I think.
 * Reciprocal best hit procedure is essentially ortholog-clustering 'lite'. If we use Orthofinder or Orthomapper it should be equivalent

SPARQL endpoints and other resources

* UniProt SPARQL: https://sparql.uniprot.org/sparql
* OrthoDB SPARQL: https://sparql.orthodb.org/ (I could not find documentation aside from their example queries, unclear how some aspects are designed)
* UniProt SPARQL tutorials: https://github.com/sib-swiss/sparql-training
* UniProt SPARQL documentation: https://sparql.uniprot.org/.well-known/void
* Review on pre-mRNA splicing: 10.1002/9780470015902.a0005037.pub3

In [18]:
import json

In [3]:
from SPARQLWrapper import SPARQLWrapper, JSON

In [13]:
uniprot = SPARQLWrapper("https://sparql.uniprot.org/sparql")

In [12]:
yeast_spliceosome_components_query = """
PREFIX taxon: <http://purl.uniprot.org/taxonomy/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX up: <http://purl.uniprot.org/core/>
PREFIX obo: <http://purl.obolibrary.org/obo/>
PREFIX odbgroup: <http://purl.orthodb.org/odbgroup/> 
SELECT DISTINCT ?protein ?genename ?taxon ?sciname ?orthogroup 
WHERE {
  ?protein a up:Protein .
  ?protein up:organism ?taxon.
  ?taxon rdfs:subClassOf taxon:4932 . # Saccharomyces cerevisiae
  ?taxon up:scientificName ?sciname .
  ?protein up:encodedBy ?gene .
  ?protein up:reviewed true .
  ?protein up:classifiedWith ?goterm .
  ?goterm rdfs:subClassOf* obo:GO_0005681 . # spliceosomal complex 
  ?gene skos:prefLabel ?genename .
  ?protein rdfs:seeAlso ?orthogroup .
  FILTER (STRSTARTS(STR(?orthogroup), STR(odbgroup:))) # OrthoDB annotation 
} ORDER BY ?genename 
"""

In [14]:
uniprot.setQuery(yeast_spliceosome_components_query)

In [15]:
uniprot.setReturnFormat(JSON)

In [16]:
yeast_spliceosome_results = uniprot.query().convert()

In [20]:
with open("yeast_spliceosome_results.json", "w") as fh:
    json.dump(yeast_spliceosome_results, fh, indent=4)

In [23]:
len(yeast_spliceosome_results['results']['bindings'])

101

In [28]:
gene2ortholog = {
    i['genename']['value'] : {
        'protein' : i['protein']['value'], 
        'odbgroup' : i['orthogroup']['value'],
    }
  for i in yeast_spliceosome_results['results']['bindings']
}

In [30]:
gene2ortholog.keys()

dict_keys(['BRR1', 'BRR2', 'BUD13', 'BUD31', 'CBC2', 'CDC40', 'CEF1', 'CLF1', 'CUS1', 'CUS2', 'CWC15', 'CWC2', 'CWC21', 'CWC22', 'CWC23', 'CWC24', 'CWC25', 'CWC27', 'DIB1', 'DRN1', 'ECM2', 'FAL1', 'HSH155', 'HSH49', 'IST3', 'ISY1', 'JIP4', 'LEA1', 'LIN1', 'LOT5', 'LSM2', 'LSM3', 'LSM4', 'LSM5', 'LSM6', 'LSM7', 'LSM8', 'LUC7', 'MAK31', 'MSL1', 'MSL5', 'MUD1', 'MUD2', 'NAM8', 'NTC20', 'NTR2', 'PML1', 'PRP11', 'PRP16', 'PRP18', 'PRP19', 'PRP2', 'PRP21', 'PRP22', 'PRP24', 'PRP28', 'PRP3', 'PRP31', 'PRP38', 'PRP39', 'PRP4', 'PRP40', 'PRP42', 'PRP43', 'PRP45', 'PRP46', 'PRP5', 'PRP6', 'PRP8', 'PRP9', 'RDS3', 'RIM4', 'RSE1', 'SAD1', 'SLU7', 'SMB1', 'SMD1', 'SMD2', 'SMD3', 'SME1', 'SMX2', 'SMX3', 'SNP1', 'SNT309', 'SNU114', 'SNU13', 'SNU23', 'SNU56', 'SNU66', 'SNU71', 'SPP2', 'SPP381', 'SPP382', 'STO1', 'SUB2', 'SYF1', 'SYF2', 'URN1', 'YHC1', 'YJU2', 'YSF3'])

In [31]:
gene2ortholog['SYF1']

{'protein': 'http://purl.uniprot.org/uniprot/Q04048',
 'odbgroup': 'http://purl.orthodb.org/odbgroup/1130725at2759'}

In [32]:
orthodb = SPARQLWrapper("https://sparql.orthodb.org/sparql")

In [35]:
ortholog_ciliate_hits = {}

In [41]:
for gene in gene2ortholog:
    odbgroup = gene2ortholog[gene]['odbgroup'].split("/")[-1] # strip prefix
    ortholog_query = """
    PREFIX odb: <http://purl.orthodb.org/>
    PREFIX og: <http://purl.orthodb.org/odbgroup/>
    
    SELECT DISTINCT ?gene ?description ?sciname ?taxon
    where {
    ?taxon rdfs:subClassOf+/up:scientificName "Ciliophora" .
    ?gene a odb:Gene;
          odb:description ?description;
          up:organism/a ?taxon .
    ?taxon up:scientificName ?sciname .
    ?gene odb:memberOf+ og:%s.
    } ORDER BY ?sciname
    """ % (odbgroup, )
    orthodb.setQuery(ortholog_query)
    orthodb.setReturnFormat(JSON)
    results = orthodb.query().convert()
    ortholog_ciliate_hits[gene] = results['results']['bindings']

Which spliceosomal components do not have an ortholog representative in ciliate genomes?

In [48]:
[gene for gene in ortholog_ciliate_hits if len(ortholog_ciliate_hits[gene]) == 0]

['BRR1',
 'CWC25',
 'JIP4',
 'LIN1',
 'LOT5',
 'LUC7',
 'MAK31',
 'NTC20',
 'NTR2',
 'PRP24',
 'PRP38',
 'RIM4',
 'SNT309',
 'SNU23',
 'SNU56',
 'SNU66',
 'SNU71',
 'SPP381',
 'SYF2',
 'URN1']

Should be interpreted with caution, as there may be related proteins/orthologs with the same function in ciliates but which are not in the same cluster as the yeast protein. For example, there are multiple ortholog groups with LUC7 related domains in OrthoDB, only one of which is in the yeast shortlist (not represented in ciliates), but others are found in ciliates.

In [49]:
import pandas as pd

In [68]:
ortholog_ciliate_hits_df = pd.DataFrame(
    [
        {'genename' : genename, 'gene' : rec['gene']['value'], 'desc' : rec['description']['value'], 'sciname' : rec['sciname']['value']}
        for genename in ortholog_ciliate_hits
        for rec in ortholog_ciliate_hits[genename]# if len(ortholog_ciliate_hits[genename]) > 0
    ]
)

In [69]:
ortholog_ciliate_hits_df.groupby(['genename','sciname'])[['genename','sciname']].value_counts().reset_index().pivot(index='genename',columns='sciname',values='count').fillna(0)

sciname,Ichthyophthirius multifiliis,Oxytricha trifallax,Paramecium tetraurelia strain d4-2,Pseudocohnilembus persalinus,Stentor coeruleus,Stylonychia lemnae,Tetrahymena thermophila SB210
genename,,,,,,,
BRR2,3.0,2.0,4.0,2.0,2.0,2.0,2.0
BUD13,1.0,1.0,2.0,1.0,1.0,1.0,1.0
BUD31,1.0,1.0,2.0,1.0,2.0,1.0,1.0
CBC2,1.0,2.0,2.0,1.0,1.0,2.0,1.0
CDC40,1.0,1.0,2.0,1.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...
SUB2,7.0,9.0,15.0,4.0,12.0,9.0,4.0
SYF1,1.0,1.0,2.0,1.0,1.0,1.0,1.0
YHC1,0.0,1.0,2.0,1.0,3.0,0.0,1.0


In [83]:
# How many orthologs represented per genome (presence/absence only, not accounting for number of copies)
ortholog_ciliate_hits_df.groupby(['sciname','genename']).size().reset_index().groupby('sciname').size()

sciname
Ichthyophthirius multifiliis          62
Oxytricha trifallax                   76
Paramecium tetraurelia strain d4-2    76
Pseudocohnilembus persalinus          67
Stentor coeruleus                     66
Stylonychia lemnae                    70
Tetrahymena thermophila SB210         77
dtype: int64

In [85]:
published_shortlist = pd.read_csv("spliceosome_shortlist.tsv",sep="\t")

How many of the yeast spliceosomal proteins we pulled from Uniprot are not in the published shortlist of 71 spliceosomal proteins from Black et al. (https://doi.org/10.1261%2Frna.079273.122)?

In [99]:
len([i for i in gene2ortholog if i not in list(published_shortlist['genename'])])

34

Which gene products named in the Black et al. shortlist are not in our list pulled from Uniprot?

In [96]:
[i for i in list(published_shortlist['genename']) if i not in gene2ortholog]

['AAR2', 'SNU40', 'SNY309', 'PRP17']

Only one of these is truly not in our shortlist, namely AAR2 (https://www.uniprot.org/uniprotkb/P32357). I'm not sure why it is not annotated with GO:0005681.

To the remainder:
 * PRP17 is a synonym for CDC40 https://www.uniprot.org/uniprotkb/P40968
 * SNU40 may be a synonym for LIN1, have to check https://www.uniprot.org/uniprotkb/P38852
 * SNY309 was not found by keyword search of Uniprot, and is probably a typo for SNT309

In [86]:
published_shortlist

,genename,group
0,LUC7,U1
1,MUD1,U1
2,NAM8,U1
3,PRP39,U1
4,SNP1,U1
...,...,...
66,PRP18,Step II
67,PRP22,Step II
68,SLU7,Step II
69,BUD13,RES
